In [1]:
import numpy as np

In [2]:
import pandas as pd

In [3]:
from validacao_externa import ExternalValidation

In [4]:
directory = "/home/jpam/Dropbox/eduardo/Projeto Cancer de Prostata/modelos/validacao_externa/"

In [5]:
dfX = pd.read_csv(directory+"X.csv",sep='\t')
y = pd.read_csv(directory+"y.txt",sep=';',header=None).values
X = dfX.values

In [6]:
test = [32,  5, 47, 21, 37, 25, 40, 13, 23, 27]

In [7]:
test.sort()

In [8]:
test

[5, 13, 21, 23, 25, 27, 32, 37, 40, 47]

In [9]:
y_test = y[test]

In [10]:
from sklearn.cross_decomposition import PLSRegression

In [11]:
train = [i for i in range(len(y)) if i not in test]

In [12]:
from cross_validation_class import CrossValidation

In [13]:
cv = CrossValidation(X[train],y[train],nLVMax=9)

In [14]:
cv.Q2()

[0.614874439229393,
 0.6090772163794136,
 0.5890934324044961,
 0.5838117601243751,
 0.58132309127919,
 0.5808970462570824,
 0.5806133262265051,
 0.5806060599521272,
 -3.4338422317241176e+30]

In [15]:
nLV = np.argmax(cv.Q2())+1

In [16]:
nLV

1

In [17]:
pls = PLSRegression(n_components = nLV)

In [18]:
pls.fit(X[train,:],y[train])

PLSRegression(copy=True, max_iter=500, n_components=1, scale=True, tol=1e-06)

In [19]:
yp = pls.predict(X[test])

In [20]:
yp

array([[7.96197275],
       [8.58978162],
       [7.35843895],
       [6.88352307],
       [7.30872159],
       [7.00626768],
       [7.48647276],
       [7.38085586],
       [8.5769546 ],
       [7.24403449]])

In [ ]:
dfPred = pd.DataFrame(data=np.c_[y_test,yp],columns=["Yreal","Ypred"])
dfPred.to_csv(directory+"previsao.csv", sep =',', index=False)

In [ ]:
np.mean(y[train])

In [ ]:
max(y[train])-min(y[train])

In [ ]:
cv.RMSECV()

In [ ]:
import math
def ssy(y,mean_y = None):
	if mean_y == None:
		mean_y = np.mean(y)
	ssy = sum((y-mean_y*np.ones(np.shape(y)))**2)
	return ssy

def calcPress(yreal,ypred):
	press = sum((yreal-ypred)**2)
	return press

def calcR2(yreal,ypred,mean_y = None):
	ssy1 = ssy(yreal,mean_y)
	R2 = 1-(calcPress(yreal,ypred)/ssy1)
	return R2

def calcMAE(yreal,ypred):
	MAE = 1/len(yreal)*np.sum(abs(yreal-ypred))
	return MAE

def calcRMSE(yreal,ypred):
	RMSE = math.sqrt(calcPress(yreal,ypred)/len(yreal))
	return RMSE


In [ ]:
ypred = yp
ytest = y_test
k = sum(ytest*ypred)/sum(ypred**2)
k1 = sum(ytest*ypred)/sum(ytest**2)
yr0 = k*ypred
y1r0 = k1*ytest
R02 = calcR2(ypred,yr0)
R102 = calcR2(ypred,y1r0)
dif = abs(R02-R102)
AREpred = sum(abs(ytest-ypred)/ytest)*100/len(ytest)
RMSEP = calcRMSE(ytest,ypred)

In [ ]:
RMSEP

In [ ]:
k

In [ ]:
k1

In [ ]:
dif

In [ ]:
AREpred

In [ ]:
dfPred = pd.DataFrame(X[train,:],columns=dfX.columns.values)

In [ ]:
dfX.columns.values

In [ ]:
dfPred.to_csv(directory+"Xtrain.csv", sep ='\t', index=False)

In [ ]:
dfPredy = pd.DataFrame(y[train,:])

In [ ]:
dfPredy.to_csv(directory+"ytrain.csv",index=False,header=False)

In [ ]:
from lno import LNO

In [ ]:
lno = LNO(X[train],y[train],1,10,5)

In [ ]:
lno.Q2

In [ ]:
desvios = np.std(lno.Q2,axis=1,ddof=1)

In [ ]:
desvios

In [ ]:
max(desvios)

In [ ]:
np.argmax(desvios)

In [ ]:
medias = np.mean(lno.Q2,axis=1)

In [ ]:
medias

In [ ]:
from plsbdg import PLSBidiag
plstrain = PLSBidiag()
plstrain.fit(X[train],y[train])
plstrain.B

In [ ]:
plstrain.indT

In [ ]:
dfLNO = pd.DataFrame(lno.Q2)

In [ ]:
dfLNO.to_csv(directory+"lno.csv")

In [ ]:
from yrandomization import YRandomization

In [ ]:
yr = YRandomization(X[train],y[train],1,50)

In [ ]:
yr.R2

In [ ]:
dfyr = pd.DataFrame(data=np.c_[yr.R2,yr.Q2,yr.R],columns=["R²","Q²","R(yrd,y)"])
dfyr.to_csv(directory+"yr.csv", sep =',', index=False)

In [21]:
ext = ExternalValidation(X,y)

In [22]:
ext.extVal(train,test)

,0
Q2F1,0.8348081839039071
Q2F2,0.8347995658898997
k,1.0026516198890347
k',0.9960048770198116
R0,0.8356473028466926
R0',0.759840351534019
dif,0.07580695131267368
AREpred,3.1557385223519328
RMSEP,0.2813625313145491
avgrm,0.7153349152807607
